In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
import os
import seaborn as sns
import pandas as pd
from collections import Counter
from sklearn.cluster import DBSCAN
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA


## Load datasets

In [ ]:
path = '/content/drive/MyDrive/cold_heads_monitoring/Dataset'
arrays = []
data_list = []

for filename in os.listdir(path):
    if filename.endswith('.pkl'):
        with open(os.path.join(path, filename), 'rb') as f:

            arr = np.asarray(pickle.load(f))[:, 1:2, :]
            arrays.append(arr)
            data_list.append(os.path.splitext(filename)[0])



In [ ]:
all_data = arrays[0]


## Balance datasets

In [ ]:
def calculate_features(window):

    mean = np.mean(window)

    std_dev = np.std(window)

    max_val = np.max(window)

    min_val = np.min(window)

    peak_to_peak = max_val - min_val

    return [mean, std_dev, max_val, min_val, peak_to_peak]


In [ ]:
def normalize_data(data):

    mean = np.mean(data, axis=0)
    std_dev = np.std(data, axis=0)

    normalized_data = (data - mean) / std_dev

    return normalized_data


In [ ]:
pca = PCA(n_components=2)

def preprocessing(signal):

    # Historical subwindows use the step stored in overlap.
    window_size = 30
    overlap = 2

    windows = []

    n = len(signal)
    indices = np.arange(0, n - window_size + 1, overlap)
    windows.extend(signal[i:i+window_size] for i in indices)

    features = np.array([calculate_features(window) for window in windows])

    return features


In [ ]:
dataset = preprocessing(all_data)[150]


## Training DBSCAN

## Inference

In [ ]:
def distance(nuovo_campione, data, clustering_labels, metric='chebyshev'):

    cluster_centroids = []
    for label in np.unique(clustering_labels):
        cluster_points = data[clustering_labels == label]
        centroid = np.mean(cluster_points, axis=0)
        cluster_centroids.append(centroid)

    cluster_centroids = np.array(cluster_centroids)

    distanze = pairwise_distances(nuovo_campione.reshape(1, -1), cluster_centroids, metric=metric)

    cluster_più_vicino = np.argmin(distanze)
    return cluster_più_vicino


In [ ]:
with open('dbscan_model.pkl', 'rb') as model_file:
    loaded_dbscan = pickle.load(model_file)


In [ ]:
data= np.expand_dims(dataset,0)
print(data.shape)

clustering_labels = loaded_dbscan.labels_

new_labels = loaded_dbscan.fit_predict(data)
